# Exploración de datos

## 1. Contexto y objetivo
El objetivo de este proyecto es analizar la producción de petróleo de pozos no convencionales de Argentina y explorar si las características de los pozos y su historial de producción permiten predecir su comportamiento productivo futuro.

La pregunta principal que buscamos responder es:

> **Dadas las características de un pozo y su historial de producción, ¿cuánto petróleo esperamos que produzca el próximo mes?**

A partir de esta pregunta se plantean tres enfoques de Machine Learning:

- **Regresión:** predecir la cantidad de petróleo que producirá un pozo durante el próximo mes.
- **Clasificación:** categorizar la producción futura del pozo en diferentes niveles de producción.
- **Clustering:** identificar grupos de pozos con características y comportamientos productivos similares.

En esta primera etapa del proyecto realizaremos una exploración de los datos disponibles, buscando comprender su estructura, calidad, variables relevantes y relaciones entre las diferentes fuentes de información.

### Importación de librerías

Para realizar el análisis exploratorio utilizaremos principalmente **Pandas**, **NumPy** y **Matplotlib**.

In [39]:
import pandas as pd
import numpy as np
import matplotlib as plt

#Indicamos que se visualicen todas las columnas del dataframe
pd.set_option("display.max_columns", None)

### Carga de los datasets

Para el análisis utilizaremos tres fuentes principales de información:

1. **Producción no convencional:** contiene registros mensuales de producción de petróleo, gas, agua y otras variables asociadas a los pozos no convencionales.
2. **Características de los pozos:** contiene información descriptiva de los pozos, como profundidad, formación, cuenca, tipo de recurso y tipo de extracción.
3. **Fracturación:** contiene información relacionada con las operaciones de fracturación de los pozos, incluyendo cantidad de fracturas, longitud de la rama horizontal, arena y agua utilizadas, entre otras variables.

Los archivos originales se mantienen sin modificar dentro de `data/raw/`.

In [6]:
produccion = pd.read_csv(
    "../data/raw/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv"
)

pozos = pd.read_csv(
    "../data/raw/capitulo-iv-pozos.csv"
)

fracturacion = pd.read_csv(
    "../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
)

## 2. Inspección inicial de los datasets

En esta primera etapa queremos conocer la estructura general de los datasets y verificar que los archivos hayan sido importados correctamente.

Antes de realizar transformaciones o eliminar datos, analizaremos:

- la cantidad de filas y columnas de cada dataset;
- los nombres de las variables disponibles;
- los tipos de datos;
- una muestra de los registros;
- la presencia de valores nulos;
- la existencia de registros duplicados.

Esta inspección inicial nos permitirá comprender qué información contiene cada fuente y detectar posibles problemas de calidad que deberán ser considerados durante las etapas posteriores del análisis.

> **Importante:** en esta etapa no eliminaremos datos automáticamente. Primero analizaremos qué representan y si los valores faltantes, ceros o duplicados tienen un significado dentro del problema.

### 2.1 Dimensiones de los datasets

Comenzamos revisando las dimensiones de cada dataset.

El atributo `shape` de Pandas devuelve una tupla con la siguiente estructura:

```text
(filas, columnas)

In [8]:
resumen_dimensiones = pd.DataFrame({
    "Dataset": [
        "Producción no convencional",
        "Pozos",
        "Fracturación"
    ],
    "Filas": [
        produccion.shape[0],
        pozos.shape[0],
        fracturacion.shape[0]
    ],
    "Columnas": [
        produccion.shape[1],
        pozos.shape[1],
        fracturacion.shape[1]
    ]
})

display(resumen_dimensiones)

,Dataset,Filas,Columnas
0,Producción no convencional,421046,40
1,Pozos,85611,26
2,Fracturación,4890,30


### 2.2. Nombres de las variables

Una vez conocidas las dimensiones de los datasets, analizamos las variables disponibles en cada fuente.

Conocer los nombres de las columnas nos permite identificar qué tipo de información contiene cada dataset y comenzar a determinar qué variables podrían ser relevantes para nuestro problema de predicción.

En particular, nos interesa identificar:

- variables que describan las características de los pozos;
- variables relacionadas con la producción;
- variables temporales, como año y mes;
- variables relacionadas con la fracturación;
- identificadores que permitan relacionar los diferentes datasets.

En esta etapa solamente identificaremos las variables disponibles.

In [15]:
columnas_datasets = pd.DataFrame({
    "Producción": pd.Series(produccion.columns),
    "Pozos": pd.Series(pozos.columns),
    "Fracturación": pd.Series(fracturacion.columns)
})

display(columnas_datasets)

,Producción,Pozos,Fracturación
0,idempresa,sigla,id_base_fractura_adjiv
1,anio,idpozo,idpozo
2,mes,area,sigla
3,idpozo,cod_area,cuenca
4,prod_pet,empresa,areapermisoconcesion
5,prod_gas,yacimiento,yacimiento
6,prod_agua,cod_yacimiento,formacion_productiva
7,iny_agua,formacion,tipo_reservorio
8,iny_gas,cuenca,subtipo_reservorio
9,iny_co2,provincia,longitud_rama_horizontal_m


### 2.3. Primer vistazo a los registros

Los nombres de las columnas nos permiten conocer qué variables están disponibles, pero todavía no muestran cómo están representados los datos.

Para obtener una primera visión del contenido de cada dataset utilizaremos `head()`, que permite visualizar los primeros registros.

Esta inspección nos ayuda a:

- comprobar que los datos fueron cargados correctamente;
- observar el formato de los valores;
- identificar posibles inconsistencias evidentes;
- relacionar los nombres de las variables con los valores que contienen.

La visualización de los primeros registros no implica que estos sean representativos de todo el dataset. Se utiliza únicamente como una primera inspección.

In [37]:
print("Primeros registros del dataset de producción:")
display(produccion.head())

Primeros registros del dataset de producción:


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,iny_otro,tef,vida_util,tipoextraccion,tipoestado,tipopozo,observaciones,fechaingreso,rectificado,habilitado,idusuario,empresa,sigla,formprod,profundidad,formacion,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,sub_tipo_recurso,fecha_data
0,YSUR,2015,1,132909,16.74,241.220,13.42,0.0,0.0,0.0,0.0,31.00,NaN,Plunger Lift,Extracción Efectiva,Gasífero,NaN,2015-02-26 13:35:35.533458,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,AEA.NQ.RCo-1039,PREC,2678.0,precuyo,SDD,AL SUR DE LA DORSAL,RQC,RANQUIL CO,NEUQUINA,Neuquén,-69.202207,-39.081820,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
1,YSUR,2018,1,132488,0.00,171.937,0.00,0.0,0.0,0.0,0.0,30.73,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2018-02-10 08:37:14.717426,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,AEA.RN.EFO-99(d),LAJA,3828.0,lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.864960,-39.016722,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
2,YSUR,2017,1,134325,22.34,365.610,1.20,0.0,0.0,0.0,0.0,31.00,NaN,Plunger Lift,Extracción Efectiva,Gasífero,NaN,2017-02-16 13:45:37.233373,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,APA.RN.EFO-117(d),LAJA,3793.0,lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.849735,-39.020644,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
3,YSUR,2018,1,132487,0.00,514.696,17.00,0.0,0.0,0.0,0.0,29.90,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2018-02-10 08:37:14.717426,f,t,444,YSUR ENERGÍA ARGENTINA S.R.L.,APA.RN.EFO-139(d),FIMP,2707.0,formación improductiva,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.837875,-39.019313,NO CONVENCIONAL,Sin Proyecto,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
4,YSUR,2016,1,153522,234.87,1651.240,94.12,0.0,0.0,0.0,0.0,30.79,NaN,Surgencia Natural,Extracción Efectiva,Gasífero,NaN,2016-02-17 10:50:46.929347,f,t,5,YSUR ENERGÍA ARGENTINA S.R.L.,APA.RN.EFO-231(d),LAJA,3844.0,lajas,FEO,ESTACION FERNANDEZ ORO,Z155,ESTACION FERNANDEZ ORO,NEUQUINA,Rio Negro,-67.810843,-39.024083,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2016-01-31


In [32]:
print("Primeros registros del dataset de pozos:")
display(pozos.head())

Primeros registros del dataset de pozos:


,sigla,idpozo,area,cod_area,empresa,yacimiento,cod_yacimiento,formacion,cuenca,provincia,cota,profundidad,clasificacion,subclasificacion,tipo_recurso,sub_tipo_recurso,gasplus,tipopozo,tipoextraccion,tipoestado,adjiv_fecha_inicio_perf,adjiv_fecha_fin_perf,adjiv_fecha_inicio_term,adjiv_fecha_fin_term,geojson,geom
0,CH.CH.EaLE.x-1,212,ESTANCIA LA ESCONDIDA,ECH,COLHUE HUAPI S.A.,ESTANCIA LA ESCONDIDA,ELA,comodoro rivadavia,GOLFO SAN JORGE,Chubut,255.00,1702.0,EXPLORACION,EXPLORACION,CONVENCIONAL,No informado,no,Petrolífero,Bombeo Mecánico,Extracción Efectiva,1996-10-30,1996-11-13,1996-11-17,1996-12-07,"{""type"":""Point"",""coordinates"":[-68.28785299999...",0101000020E61000008D43FD2E6C1251C00E4B033FAACB...
1,CH.CH.EaLE.x-2,213,ESTANCIA LA ESCONDIDA,ECH,COLHUE HUAPI S.A.,ESTANCIA LA ESCONDIDA,ELA,comodoro rivadavia,GOLFO SAN JORGE,Chubut,251.00,1350.0,EXPLORACION,EXPLORACION,CONVENCIONAL,No informado,no,Inyección de Agua,Sin Sistema de Extracción,En Inyección Efectiva,1996-10-14,1996-11-13,1996-12-09,1996-12-17,"{""type"":""Point"",""coordinates"":[-68.29201899999...",0101000020E6100000E1B37570B01251C00BB5A679C7CB...
2,CH.CH.EaLE-3,214,ESTANCIA LA ESCONDIDA,ECH,COLHUE HUAPI S.A.,ESTANCIA LA ESCONDIDA,ELA,comodoro rivadavia,GOLFO SAN JORGE,Chubut,256.15,1350.0,EXPLOTACION,DESARROLLO,CONVENCIONAL,No informado,no,Petrolífero,Bombeo Mecánico,Extracción Efectiva,1997-01-30,1997-02-07,1997-03-02,1997-03-10,"{""type"":""Point"",""coordinates"":[-68.28387800000...",0101000020E61000007383A10E2B1251C082548A1D8DCB...
3,CH.CH.EaLE-4,215,ESTANCIA LA ESCONDIDA,ECH,COLHUE HUAPI S.A.,ESTANCIA LA ESCONDIDA,ELA,comodoro rivadavia,GOLFO SAN JORGE,Chubut,256.70,1351.0,EXPLOTACION,DESARROLLO,CONVENCIONAL,No informado,no,Petrolífero,Bombeo Mecánico,Extracción Efectiva,1997-01-13,1997-01-20,1997-01-23,1997-02-07,"{""type"":""Point"",""coordinates"":[-68.28948300000...",0101000020E6100000DF6FB4E3861251C0A148F7730ACC...
4,CH.CH.EaLE-5,216,ESTANCIA LA ESCONDIDA,ECH,COLHUE HUAPI S.A.,ESTANCIA LA ESCONDIDA,ELA,comodoro rivadavia,GOLFO SAN JORGE,Chubut,256.00,1350.0,EXPLOTACION,DESARROLLO,CONVENCIONAL,No informado,no,Petrolífero,Bombeo Mecánico,Extracción Efectiva,1997-02-16,1997-02-24,1997-03-10,1997-03-18,"{""type"":""Point"",""coordinates"":[-68.29447500000...",0101000020E6100000569FABADD81251C0B98D06F016CC...


In [33]:
print("Primeros registros del dataset de fracturación:")
display(fracturacion.head())

Primeros registros del dataset de fracturación:


,id_base_fractura_adjiv,idpozo,sigla,cuenca,areapermisoconcesion,yacimiento,formacion_productiva,tipo_reservorio,subtipo_reservorio,longitud_rama_horizontal_m,cantidad_fracturas,tipo_terminacion,arena_bombeada_nacional_tn,arena_bombeada_importada_tn,agua_inyectada_m3,co2_inyectado_m3,presion_maxima_psi,potencia_equipos_fractura_hp,fecha_inicio_fractura,fecha_fin_fractura,fecha_data,anio_if,mes_if,anio_ff,mes_ff,anio_carga,mes_carga,empresa_informante,mes,anio
0,30,159910,APS.Nq.ADC.xp-1033,NEUQUINA,AGUA DEL CAJON,AGUA DEL CAJON,los molles,NO CONVENCIONAL,SHALE,0.0,3,Punzado,0.000,0.000,2718.20,0.0,10190.0,10897.0,2019-04-20,2019-04-30,2019-06-14 17:13:03.68279,2019,4,2019,4,2019,6,CAPEX S.A.,4,2019
1,31,159910,APS.Nq.ADC.xp-1033,NEUQUINA,AGUA DEL CAJON,AGUA DEL CAJON,los molles,NO CONVENCIONAL,SHALE,0.0,1,Punzado,0.000,0.000,600.00,0.0,9250.0,10251.0,2018-11-02,2018-11-03,2019-06-14 17:14:19.179874,2018,11,2018,11,2019,6,CAPEX S.A.,11,2018
2,37,159219,YPF.Nq.AdlA-1001(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1437.3,18,Tapón disparo,3761.370,536.850,25768.30,0.0,15000.0,32000.0,2017-11-19,2017-12-14,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017
3,38,159220,YPF.Nq.AdlA-1002(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1518.3,19,Tapón disparo,3903.705,558.225,27398.37,0.0,11348.0,32000.0,2017-11-21,2017-12-15,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017
4,39,159221,YPF.Nq.AdlA-1003(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1482.3,19,Tapón disparo,3949.020,569.925,27157.60,0.0,11076.0,32000.0,2017-11-18,2017-12-15,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017


### 2.4. Tipos de datos

A continuación analizamos el tipo de dato asignado por Pandas a cada variable.

Los tipos de datos son importantes porque determinan qué operaciones podemos realizar sobre cada columna.

Por ejemplo:

- las variables numéricas pueden utilizarse para cálculos estadísticos;
- las variables de texto permiten agrupar o analizar categorías;
- las variables de fecha requieren un tratamiento específico para realizar análisis temporales.

Esta revisión también nos permite detectar columnas que eventualmente deberán ser transformadas antes del análisis o del entrenamiento de los modelos.

In [38]:
print("Tipos de datos - Producción:")
display(produccion.dtypes)

Tipos de datos - Producción:


idempresa                     str
anio                        int64
mes                         int64
idpozo                      int64
prod_pet                  float64
prod_gas                  float64
prod_agua                 float64
iny_agua                  float64
iny_gas                   float64
iny_co2                   float64
iny_otro                  float64
tef                       float64
vida_util                 float64
tipoextraccion                str
tipoestado                    str
tipopozo                      str
observaciones                 str
fechaingreso                  str
rectificado                   str
habilitado                    str
idusuario                   int64
empresa                       str
sigla                         str
formprod                      str
profundidad               float64
formacion                     str
idareapermisoconcesion        str
areapermisoconcesion          str
idareayacimiento              str
areayacimiento

In [20]:
print("Tipos de datos - Pozos:")
display(pozos.dtypes)

Tipos de datos - Pozos:


sigla                          str
idpozo                       int64
area                           str
cod_area                       str
empresa                        str
yacimiento                     str
cod_yacimiento                 str
formacion                      str
cuenca                         str
provincia                      str
cota                       float64
profundidad                float64
clasificacion                  str
subclasificacion               str
tipo_recurso                   str
sub_tipo_recurso               str
gasplus                        str
tipopozo                       str
tipoextraccion                 str
tipoestado                     str
adjiv_fecha_inicio_perf        str
adjiv_fecha_fin_perf           str
adjiv_fecha_inicio_term        str
adjiv_fecha_fin_term           str
geojson                        str
geom                           str
dtype: object

In [21]:
print("Tipos de datos - Fracturación:")
display(fracturacion.dtypes)

Tipos de datos - Fracturación:


id_base_fractura_adjiv            int64
idpozo                            int64
sigla                               str
cuenca                              str
areapermisoconcesion                str
yacimiento                          str
formacion_productiva                str
tipo_reservorio                     str
subtipo_reservorio                  str
longitud_rama_horizontal_m      float64
cantidad_fracturas                int64
tipo_terminacion                    str
arena_bombeada_nacional_tn      float64
arena_bombeada_importada_tn     float64
agua_inyectada_m3               float64
co2_inyectado_m3                float64
presion_maxima_psi              float64
potencia_equipos_fractura_hp    float64
fecha_inicio_fractura               str
fecha_fin_fractura                  str
fecha_data                          str
anio_if                           int64
mes_if                            int64
anio_ff                           int64
mes_ff                            int64


### 2.5. Información general de los datasets

Utilizamos el método `info()` para obtener un resumen general de cada DataFrame.

Esta función permite observar:

- cantidad de registros;
- cantidad de columnas;
- nombre de las variables;
- cantidad de valores no nulos;
- tipo de dato de cada variable;
- uso aproximado de memoria.

Esta información será útil para detectar variables con valores faltantes y posibles diferencias entre los tipos de datos esperados y los encontrados.

In [ ]:
print("Resumen general - Producción:")
produccion.info()

<class 'pandas.DataFrame'>
RangeIndex: 421046 entries, 0 to 421045
Data columns (total 40 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   idempresa               421046 non-null  str    
 1   anio                    421046 non-null  int64  
 2   mes                     421046 non-null  int64  
 3   idpozo                  421046 non-null  int64  
 4   prod_pet                421046 non-null  float64
 5   prod_gas                421046 non-null  float64
 6   prod_agua               421046 non-null  float64
 7   iny_agua                421046 non-null  float64
 8   iny_gas                 421046 non-null  float64
 9   iny_co2                 421046 non-null  float64
 10  iny_otro                421046 non-null  float64
 11  tef                     421046 non-null  float64
 12  vida_util               8964 non-null    float64
 13  tipoextraccion          420432 non-null  str    
 14  tipoestado              420432 

In [24]:
print("Resumen general - Pozos:")
pozos.info()

Resumen general - Pozos:
<class 'pandas.DataFrame'>
RangeIndex: 85611 entries, 0 to 85610
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   sigla                    85611 non-null  str    
 1   idpozo                   85611 non-null  int64  
 2   area                     85611 non-null  str    
 3   cod_area                 85611 non-null  str    
 4   empresa                  84647 non-null  str    
 5   yacimiento               85611 non-null  str    
 6   cod_yacimiento           85611 non-null  str    
 7   formacion                82796 non-null  str    
 8   cuenca                   85611 non-null  str    
 9   provincia                85611 non-null  str    
 10  cota                     85611 non-null  float64
 11  profundidad              85611 non-null  float64
 12  clasificacion            85611 non-null  str    
 13  subclasificacion         85611 non-null  str    
 14  tipo_rec

In [25]:
print("Resumen general - Fracturación:")
fracturacion.info()

Resumen general - Fracturación:
<class 'pandas.DataFrame'>
RangeIndex: 4890 entries, 0 to 4889
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_base_fractura_adjiv        4890 non-null   int64  
 1   idpozo                        4890 non-null   int64  
 2   sigla                         4890 non-null   str    
 3   cuenca                        4890 non-null   str    
 4   areapermisoconcesion          4890 non-null   str    
 5   yacimiento                    4890 non-null   str    
 6   formacion_productiva          4890 non-null   str    
 7   tipo_reservorio               4855 non-null   str    
 8   subtipo_reservorio            3916 non-null   str    
 9   longitud_rama_horizontal_m    4890 non-null   float64
 10  cantidad_fracturas            4890 non-null   int64  
 11  tipo_terminacion              4890 non-null   str    
 12  arena_bombeada_nacional_tn    4890 non-nu

### 2.6. Análisis de valores faltantes

Los valores faltantes son registros en los que una variable no contiene información.

Su presencia no implica necesariamente un error en los datos. En algunos casos puede significar que una característica no fue informada, que no corresponde al registro o que la fuente original no disponía de ese dato.

Por este motivo, en esta etapa no eliminaremos automáticamente las filas o columnas con valores faltantes. Primero identificaremos su cantidad y proporción para evaluar posteriormente cómo tratarlos.

Calcularemos:

- cantidad de valores faltantes por variable;
- porcentaje de valores faltantes respecto del total de registros.

In [26]:
def resumen_nulos(df):
    resumen = pd.DataFrame({
        "Valores nulos": df.isnull().sum(),
        "Porcentaje nulos": df.isnull().mean() * 100
    })
    
    return resumen.sort_values(
        by="Porcentaje nulos",
        ascending=False
    )

In [27]:
print("Valores faltantes - Producción")
display(resumen_nulos(produccion))

Valores faltantes - Producción


,Valores nulos,Porcentaje nulos
vida_util,412082,97.871016
observaciones,397045,94.299673
clasificacion,922,0.218978
subclasificacion,922,0.218978
tipopozo,614,0.145827
tipoextraccion,614,0.145827
tipoestado,614,0.145827
sub_tipo_recurso,452,0.107352
iny_agua,0,0.000000
prod_agua,0,0.000000


In [28]:
print("Valores faltantes - Pozos")
display(resumen_nulos(pozos))

Valores faltantes - Pozos


,Valores nulos,Porcentaje nulos
adjiv_fecha_inicio_term,36487,42.619523
adjiv_fecha_fin_term,36486,42.618355
adjiv_fecha_fin_perf,34149,39.888566
adjiv_fecha_inicio_perf,34004,39.719195
formacion,2815,3.288129
empresa,964,1.126024
idpozo,0,0.000000
sigla,0,0.000000
cod_area,0,0.000000
area,0,0.000000


In [29]:
print("Valores faltantes - Fracturación")
display(resumen_nulos(fracturacion))

Valores faltantes - Fracturación


,Valores nulos,Porcentaje nulos
subtipo_reservorio,974,19.918200
potencia_equipos_fractura_hp,51,1.042945
tipo_reservorio,35,0.715746
sigla,0,0.000000
id_base_fractura_adjiv,0,0.000000
cuenca,0,0.000000
yacimiento,0,0.000000
areapermisoconcesion,0,0.000000
formacion_productiva,0,0.000000
longitud_rama_horizontal_m,0,0.000000


#### 2.6.1. Valores faltantes en el dataset de producción

El dataset de producción presenta una cobertura completa para las principales variables productivas.

En particular, las variables `prod_pet`, `prod_gas`, `prod_agua`, `idpozo`, `anio` y `mes` no presentan valores faltantes.

La variable `vida_util` presenta un porcentaje muy elevado de valores ausentes (97,87%), mientras que `observaciones` presenta un 94,30% de valores ausentes. Debido a esta baja cobertura, estas variables deberán analizarse antes de considerar su utilización en etapas posteriores.

El resto de las variables con valores faltantes presenta porcentajes inferiores al 0,25%, por lo que su impacto sobre el conjunto de datos es reducido.

Por el momento no se eliminarán estas variables ni se imputarán sus valores. Su tratamiento se definirá posteriormente en función de su utilidad para el análisis y los modelos.

#### 2.6.2. Valores faltantes en el dataset de pozos

En el dataset de características de los pozos se observa una cobertura completa en la mayoría de las variables.

Las principales ausencias se concentran en las variables relacionadas con las fechas de perforación y terminación:

- `adjiv_fecha_inicio_term`: 42,62% de valores faltantes.
- `adjiv_fecha_fin_term`: 42,62%.
- `adjiv_fecha_fin_perf`: 39,89%.
- `adjiv_fecha_inicio_perf`: 39,72%.

También se observan valores faltantes en `formacion` (3,29%) y `empresa` (1,13%).

Los valores faltantes en las fechas deberán analizarse considerando que no necesariamente representan un error: pueden corresponder a pozos para los cuales determinada información no fue registrada o no se encuentra disponible en la fuente.

Por lo tanto, estas variables no serán eliminadas automáticamente y su tratamiento se definirá según su utilización en el análisis posterior.

#### 2.6.3. Valores faltantes en el dataset de fracturación

El dataset de fracturación presenta una cobertura elevada en la mayoría de sus variables.

La principal excepción es `subtipo_reservorio`, que presenta un 19,92% de valores faltantes. También se observan valores ausentes en `potencia_equipos_fractura_hp` (1,04%) y `tipo_reservorio` (0,72%).

Las variables directamente relacionadas con las características de la fracturación, como `longitud_rama_horizontal_m`, `cantidad_fracturas`, `arena_bombeada_nacional_tn`, `arena_bombeada_importada_tn`, `agua_inyectada_m3` y `presion_maxima_psi`, no presentan valores faltantes.

Esto resulta especialmente relevante para nuestro análisis, ya que estas variables podrían utilizarse posteriormente para caracterizar los pozos y estudiar su relación con el comportamiento productivo.

#### 2.6.4. Conclusiones sobre los valores faltantes

El análisis muestra que los tres datasets presentan, en general, una alta cobertura de información.

Las principales situaciones detectadas son:

- En **producción**, `vida_util` y `observaciones` presentan una proporción muy elevada de valores faltantes, mientras que las variables productivas principales no presentan ausencias.
- En **pozos**, los valores faltantes se concentran principalmente en las fechas de perforación y terminación.
- En **fracturación**, `subtipo_reservorio` presenta la mayor proporción de valores faltantes, mientras que las principales variables cuantitativas relacionadas con la fracturación presentan información completa.

En consecuencia, no se realizará una eliminación general de valores faltantes. El tratamiento será definido posteriormente según el uso de cada variable y las necesidades de los modelos.

Además, se tendrá especial cuidado en diferenciar los valores faltantes (`NaN`) de los valores cero, ya que un cero puede representar una medición real de producción o de otra variable.

### 2.7. Análisis de registros duplicados

Antes de realizar transformaciones, verificaremos si existen registros duplicados en las diferentes fuentes de datos.

Es importante diferenciar entre un registro completamente duplicado y la repetición de un identificador como `idpozo`.

En el dataset de producción es esperable que un mismo `idpozo` aparezca varias veces, ya que cada registro representa la producción de un pozo en un determinado período.

Por lo tanto, la repetición de `idpozo` no será considerada por sí sola como un duplicado. Primero analizaremos los registros completamente repetidos y luego la cantidad de registros y períodos disponibles para cada pozo.

In [44]:
resumen_duplicados = pd.DataFrame({
    "Dataset": [
        "Producción no convencional",
        "Pozos",
        "Fracturación"
    ],
    "Registros duplicados": [
        produccion.duplicated().sum(),
        pozos.duplicated().sum(),
        fracturacion.duplicated().sum()
    ]
})

display(resumen_duplicados)

,Dataset,Registros duplicados
0,Producción no convencional,0
1,Pozos,0
2,Fracturación,0


#### Conclusiones sobre los registros duplicados

No se encontraron registros completamente duplicados en ninguno de los tres datasets analizados.

Por lo tanto, no es necesario realizar una eliminación de duplicados en esta etapa.

La repetición de `idpozo` se analizará posteriormente de acuerdo con la granularidad de cada dataset, ya que un mismo pozo puede estar asociado a múltiples registros correspondientes a distintos períodos de producción o eventos de fracturación.
